In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd


# ============================================================================
# 11. LEAKAGE / SHORTCUT AUDIT: LITERAL COUNTRY & LANGUAGE NAME MENTIONS
# ============================================================================
# This checks a different, more concrete question than doc_id/filename
# leakage (which is ruled out by code inspection -- doc_id is bookkeeping
# only, never sent to the SVM's feature extractor or the LLM prompt in
# 03/05/07/08/09). This notebook checks whether the ESSAY TEXT ITSELF
# contains literal self-disclosure of the writer's country/nationality/
# language -- exactly the "shortcut" mechanism Uluslu & Schneider (2025)
# describe LLMs exploiting instead of doing real linguistic analysis. If an
# essay literally says "In Spain, ..." or "as a Spanish speaker...", a model
# can get the right answer without any structural analysis at all, which
# would inflate full-text accuracy in a way that doesn't reflect genuine
# NLI capability.
#
# This is a heuristic regex screen, not a definitive leakage proof. Some
# terms are ambiguous out of context -- "turkey" the bird vs. Turkey the
# country is the clearest case, handled below by requiring a capital T for
# that language only. Treat flagged essays as candidates for manual
# spot-checking, not automatic proof of a shortcut.

BASE_DIR = Path("..").resolve()
COMBINED_CSV_PATH = BASE_DIR / "data/processed/combined_nli_masked.csv"
FULLTEXT_LLM_PATH = BASE_DIR / "data/processed/results/llm_predictions_fulltext.csv"
RESULTS_DIR = BASE_DIR / "data/processed/results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_FLAGGED_PATH = RESULTS_DIR / "leakage_flagged_essays.csv"
OUTPUT_SUMMARY_PATH = RESULTS_DIR / "leakage_summary_by_partition_label.csv"
OUTPUT_ACCURACY_SPLIT_PATH = RESULTS_DIR / "leakage_accuracy_comparison.csv"
OUTPUT_SVM_ACCURACY_SPLIT_PATH = RESULTS_DIR / "leakage_svm_accuracy_comparison.csv"

# Case-insensitive terms, except TUR which requires a capital "Turkey" / "Turkish" / "Turks"
# to avoid matching "turkey" the bird.
LANG_TERMS_CASE_INSENSITIVE = {
    "DEU": ["germany", "german", "germans", "deutschland"],
    "FRA": ["france", "french", "frenchman", "frenchwoman", "frenchmen", "frenchwomen"],
    "ITA": ["italy", "italian", "italians"],
    "JPN": ["japan", "japanese"],
    "KOR": ["korea", "korean", "koreans", "south korea", "north korea"],
    "SPA": ["spain", "spanish", "spaniard", "spaniards"],
    "ZHO": ["china", "chinese", "mandarin chinese", "cantonese"],
}
LANG_TERMS_CASE_SENSITIVE = {
    "TUR": ["Turkey", "Turkish", "Turks", "Türkiye", "Turkiye"],
}


def build_patterns() -> dict:
    patterns = {}
    for label, terms in LANG_TERMS_CASE_INSENSITIVE.items():
        patterns[label] = re.compile(r"\b(" + "|".join(re.escape(t) for t in terms) + r")\b", re.IGNORECASE)
    for label, terms in LANG_TERMS_CASE_SENSITIVE.items():
        patterns[label] = re.compile(r"\b(" + "|".join(re.escape(t) for t in terms) + r")\b")
    return patterns


PATTERNS = build_patterns()


def find_matches(text: str) -> dict:
    """Return {label: sorted unique matched terms} for every label with a hit in `text`."""
    hits = {}
    for label, pattern in PATTERNS.items():
        found = sorted(set(m.group(0) for m in pattern.finditer(text)))
        if found:
            hits[label] = found
    return hits


print("Loading combined corpus...")
df = pd.read_csv(COMBINED_CSV_PATH)
df["text"] = df["text"].fillna("").astype(str)
df["masked_text"] = df["masked_text"].fillna("").astype(str)
df["label"] = df["label"].astype(str).str.strip().str.upper()
df["dataset"] = df["dataset"].astype(str).str.strip().str.upper()
df["split"] = df["split"].astype(str).str.strip().str.lower()

records = []
for _, row in df.iterrows():
    text_hits = find_matches(row["text"])
    masked_hits = find_matches(row["masked_text"])
    if text_hits or masked_hits:
        records.append(
            {
                "doc_id": row["doc_id"],
                "dataset": row["dataset"],
                "label": row["label"],
                "matched_labels_in_text": ",".join(sorted(text_hits)),
                "matched_terms_in_text": "; ".join(f"{k}:{','.join(v)}" for k, v in text_hits.items()),
                "self_referential_in_text": row["label"] in text_hits,
                "matched_labels_in_masked_text": ",".join(sorted(masked_hits)),
                "self_referential_survived_masking": row["label"] in masked_hits,
            }
        )

flagged = pd.DataFrame(records)
flagged.to_csv(OUTPUT_FLAGGED_PATH, index=False)

print(f"Flagged {len(flagged)} / {len(df)} essays with ANY country/language mention (self- or cross-referential).")
if len(flagged):
    n_self = int(flagged["self_referential_in_text"].sum())
    print(f"Of those, {n_self} mention their OWN label's country/language -- the concerning case.")
    n_survived = int(flagged["self_referential_survived_masking"].sum())
    if n_survived:
        print(
            f"WARNING: {n_survived} self-referential mentions SURVIVED POS masking -- spaCy did not tag "
            "them as PROPN/ADJ, so they also leak into the masked-text condition, not just full-text."
        )
    else:
        print("None of the self-referential mentions survived POS masking -- the masked condition looks clean of this specific leak.")
else:
    print("No country/language name mentions found at all -- either the corpus is clean of this, or the term list needs expanding.")

print(f"Saved: {OUTPUT_FLAGGED_PATH}")

# --- Summary by partition x label: rate of self-referential mentions ---
conditions = [
    df["dataset"] == "ICLEV3",
    (df["dataset"] == "TOEFL11") & (df["split"] == "test"),
    (df["dataset"] == "TOEFL11") & (df["split"] == "train"),
]
choices = ["iclev3_external_test", "toefl11_test", "toefl11_train"]
df["partition"] = np.select(conditions, choices, default="other")

flagged_self_ids = set(flagged.loc[flagged["self_referential_in_text"], "doc_id"]) if len(flagged) else set()
df["self_referential_in_text"] = df["doc_id"].isin(flagged_self_ids)

summary = (
    df.groupby(["partition", "label"])["self_referential_in_text"]
    .agg(n="size", n_flagged="sum")
    .assign(pct_flagged=lambda d: (d["n_flagged"] / d["n"] * 100).round(1))
    .reset_index()
)
summary.to_csv(OUTPUT_SUMMARY_PATH, index=False)
print("\n=== Self-referential mention rate by partition x label (%) ===")
print(summary.to_string(index=False))
print(f"Saved: {OUTPUT_SUMMARY_PATH}")

# --- Does the LLM do better on essays that self-disclose their country/language? ---
if FULLTEXT_LLM_PATH.exists():
    llm_df = pd.read_csv(FULLTEXT_LLM_PATH)
    llm_df = llm_df[llm_df["error"].fillna("").eq("") & llm_df["predicted_l1"].notna()].copy()

    if len(llm_df) < 800:
        print(
            f"\nNOTE: {FULLTEXT_LLM_PATH.name} has only {len(llm_df)} rows, not the full 800 -- "
            "run 08_llm_full_text_baseline.ipynb to completion (RUN_FULL_EVALUATION = True) for a "
            "reliable accuracy-split comparison. Proceeding with what's available."
        )

    llm_df = llm_df.merge(df[["doc_id", "self_referential_in_text"]], on="doc_id", how="left")
    llm_df["correct"] = llm_df["predicted_l1"] == llm_df["true_l1"]

    # Break out by partition (TOEFL11 vs ICLE), same as the SVM check below, instead of only
    # reporting a pooled "all" number. Self-disclosure isn't ICLE-specific -- TOEFL11 test essays
    # get flagged too -- so a pooled number can't tell you whether the shortcut effect differs in
    # size between the two corpora, which matters for the paper's TOEFL11-anchored argument.
    llm_acc_rows = []
    if "eval_partition" in llm_df.columns:
        for partition, part_df in llm_df.groupby("eval_partition"):
            split = part_df.groupby("self_referential_in_text")["correct"].agg(n="size", accuracy="mean").reset_index()
            split.insert(0, "partition", partition)
            llm_acc_rows.append(split)
    else:
        print(f"NOTE: {FULLTEXT_LLM_PATH.name} has no eval_partition column -- skipping the per-partition breakdown.")

    overall = llm_df.groupby("self_referential_in_text")["correct"].agg(n="size", accuracy="mean").reset_index()
    overall.insert(0, "partition", "all")
    llm_acc_rows.append(overall)

    acc_split = pd.concat(llm_acc_rows, ignore_index=True)
    acc_split.to_csv(OUTPUT_ACCURACY_SPLIT_PATH, index=False)

    print("\n=== Full-text LLM accuracy: essays WITH vs. WITHOUT a self-referential country/language mention (by partition) ===")
    print(acc_split.to_string(index=False))
    print(f"Saved: {OUTPUT_ACCURACY_SPLIT_PATH}")
    print(
        "\nIf accuracy is meaningfully higher on the 'True' row, that's direct evidence the LLM is using "
        "literal self-disclosure as a shortcut on at least part of the full-text condition -- not proof "
        "for every correct prediction, but a real, checkable signal rather than a hypothetical one. Compare "
        "the toefl11_test and iclev3_external_test rows to see whether the shortcut effect differs by corpus."
    )
else:
    print(f"\nSkipping the LLM accuracy split -- {FULLTEXT_LLM_PATH.name} not found. Run 08 first.")

# --- Does the SVM do better on essays that self-disclose their country/language? ---
# Full-text SVM (07) uses word/char TF-IDF, which can pick up literal country/language
# names as strong features just as easily as an LLM reads them in its prompt. The masked
# SVM (04) should already be mostly protected from this by POS masking (see the
# masking-survival check above: only 2/2012 self-referential mentions leaked through).
# Comparing the two tells us whether the earlier full-text-vs-masked SVM accuracy delta
# (07 vs 04: +14.6pp TOEFL11, +16.1pp ICLEv3) is genuine lexical/structural signal or
# partly this same leak -- reusing the self_referential_in_text flag already computed
# above rather than re-deriving it, so this stays consistent with the LLM check.
SVM_SOURCES = [
    ("SVM_masked", RESULTS_DIR / "svm_predictions_full.csv", "svm_predicted_l1"),
    ("SVM_fulltext", RESULTS_DIR / "svm_predictions_fulltext.csv", "svm_predicted_l1"),
]

svm_accuracy_rows = []
for system_name, path, pred_col in SVM_SOURCES:
    if not path.exists():
        print(f"\nSkipping {system_name}: {path.name} not found (run 04 and/or 07 first).")
        continue

    svm_df = pd.read_csv(path)
    svm_df = svm_df.merge(df[["doc_id", "self_referential_in_text"]], on="doc_id", how="left")
    svm_df["correct"] = svm_df[pred_col] == svm_df["true_l1"]

    for partition, part_df in svm_df.groupby("partition"):
        split = part_df.groupby("self_referential_in_text")["correct"].agg(n="size", accuracy="mean").reset_index()
        split.insert(0, "partition", partition)
        split.insert(0, "system", system_name)
        svm_accuracy_rows.append(split)

    overall = svm_df.groupby("self_referential_in_text")["correct"].agg(n="size", accuracy="mean").reset_index()
    overall.insert(0, "partition", "all")
    overall.insert(0, "system", system_name)
    svm_accuracy_rows.append(overall)

if svm_accuracy_rows:
    svm_accuracy_split = pd.concat(svm_accuracy_rows, ignore_index=True)
    svm_accuracy_split.to_csv(OUTPUT_SVM_ACCURACY_SPLIT_PATH, index=False)
    print("\n=== SVM accuracy: essays WITH vs. WITHOUT a self-referential country/language mention ===")
    print(svm_accuracy_split.to_string(index=False))
    print(f"Saved: {OUTPUT_SVM_ACCURACY_SPLIT_PATH}")
    print(
        "\nCompare SVM_masked's gap (should be small -- masking already strips most of this) against "
        "SVM_fulltext's gap. If SVM_fulltext's gap is large, the earlier 14.6-16pp full-text-vs-masked "
        "SVM delta is partly this same leak, not purely richer lexical/structural modeling."
    )
else:
    print("\nNo SVM prediction files found -- run 04_model_training.ipynb and 07_unmasked_svm_baseline.ipynb first.")
